# E-commerce Sales Analysis
## Brazilian E-Commerce Public Dataset by Olist

This notebook performs comprehensive analysis of e-commerce data including:
- **Sales Performance** - Revenue trends, top categories
- **Customer Behavior** - Demographics, segmentation, lifetime value
- **Product Analysis** - Category performance, affinity analysis
- **Delivery Efficiency** - Delivery times, delays, impact on reviews
- **Payment Patterns** - Payment methods, installment usage
- **KPI Dashboard** - Key performance indicators
- **Advanced Insights** - Cohort analysis, churn risk, forecasting
- **Interactive Dashboards** - Plotly-based visualizations

## Part 1: Setup and Configuration

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import sys
import warnings
warnings.filterwarnings('ignore')

# Add src directory to path
sys.path.append(os.path.join(os.getcwd(), '..'))

# Import our custom modules
from src.data_cleaning import load_data, clean_data, merge_data
from src.analysis import (
    analyze_sales,
    analyze_customers,
    analyze_products,
    analyze_delivery,
    analyze_payments,
    generate_insights,
    generate_full_analysis
)
from src.visualization import save_all_charts
from src.kpi_dashboard import calculate_kpis, generate_kpi_summary, get_kpi_alerts, create_kpi_scorecard
from src.advanced_insights import (
    perform_rfm_segmentation,
    calculate_customer_lifetime_value,
    forecast_sales,
    analyze_product_affinity,
    calculate_churn_risk,
    generate_advanced_insights_report
)
from src.interactive_dashboard import create_full_dashboard
from src.executive_report import generate_executive_summary, save_executive_report

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.float_format', '{:.2f}'.format)

# Set plotting style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

print('All libraries imported successfully!')

## Part 2: Data Loading

In [ ]:
# Define paths
DATA_DIR = '../data'
OUTPUT_DIR = '../outputs'

# Load all datasets
print('=' * 60)
print('LOADING DATASETS')
print('=' * 60)

raw_data = load_data(DATA_DIR)
print(f'\nLoaded {len(raw_data)} datasets')

### Dataset Overview

In [ ]:
# Display basic info for each dataset
for name, df in raw_data.items():
    if df is not None:
        print(f'\n{name.upper()} Dataset:')
        print(f'  Shape: {df.shape[0]:,} rows x {df.shape[1]} columns')
        print(f'  Columns: {list(df.columns)[:8]}...' if len(df.columns) > 8 else f'  Columns: {list(df.columns)}')

## Part 3: Data Cleaning

In [ ]:
# Clean all datasets
print('=' * 60)
print('CLEANING DATA')
print('=' * 60)

cleaned = clean_data(raw_data)
print('\nData cleaning complete!')

### Missing Values Summary

In [ ]:
# Check missing values in key datasets
print('Missing Values in Orders Dataset:')
print(cleaned['orders'].isnull().sum())

print('\nMissing Values in Products Dataset:')
print(cleaned['products'].isnull().sum())

## Part 4: Data Merging

In [ ]:
# Merge all datasets
print('=' * 60)
print('MERGING DATASETS')
print('=' * 60)

df = merge_data(cleaned)

print(f'\nFinal merged dataset: {df.shape[0]:,} rows, {df.shape[1]} columns')

In [ ]:
# Display sample of merged data
print('\nSample of Merged Data (first 3 rows):')
df.head(3)

## Part 5: KPI Dashboard

In [ ]:
# Calculate comprehensive KPIs
print('=' * 60)
print('KPI DASHBOARD')
print('=' * 60)

kpis = calculate_kpis(df)

# Display KPI summary
print(generate_kpi_summary(kpis))

In [ ]:
# Display KPI alerts
alerts = get_kpi_alerts(kpis)

if alerts:
    print('\n' + '=' * 60)
    print('KPI ALERTS')
    print('=' * 60)
    for alert in alerts:
        icon = '[!!]' if alert['severity'] == 'HIGH' else '[!]'
        print(f"\n{icon} {alert['severity']}: {alert['message']}")
else:
    print('\nNo critical alerts - all KPIs within healthy ranges!')

In [ ]:
# Create KPI scorecard
scorecard = create_kpi_scorecard(kpis)
print('\nKPI Scorecard:')
scorecard

## Part 6: Sales Analysis

In [ ]:
# Run sales analysis
print('=' * 60)
print('SALES ANALYSIS')
print('=' * 60)

sales_results = analyze_sales(df)

print(f"\nTotal Revenue: ${sales_results['total_revenue']:,.2f}")
print(f"Total Orders: {sales_results['total_orders']:,}")
print(f"Average Order Value: ${sales_results['avg_order_value']:.2f}")

In [ ]:
# Monthly Revenue Trend
if not sales_results['monthly_revenue'].empty:
    plt.figure(figsize=(14, 6))
    monthly = sales_results['monthly_revenue']
    plt.plot(monthly['month'].astype(str), monthly['revenue'], 
             marker='o', linewidth=2, markersize=8, color='#2E86AB')
    plt.xlabel('Month', fontsize=12)
    plt.ylabel('Revenue ($)', fontsize=12)
    plt.title('Monthly Revenue Trend', fontsize=14, fontweight='bold')
    plt.xticks(rotation=45)
    plt.gca().yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x:,.0f}'))
    plt.tight_layout()
    plt.show()

In [ ]:
# Top Categories by Revenue
if not sales_results['top_categories'].empty:
    plt.figure(figsize=(12, 8))
    top_cats = sales_results['top_categories'].head(10)
    
    colors = sns.color_palette('viridis', len(top_cats))
    bars = plt.barh(range(len(top_cats)), top_cats['revenue'], color=colors)
    
    plt.yticks(range(len(top_cats)), 
               [cat.replace('_', ' ').title() for cat in top_cats['category']])
    plt.xlabel('Revenue ($)', fontsize=12)
    plt.title('Top 10 Product Categories by Revenue', fontsize=14, fontweight='bold')
    plt.gca().invert_yaxis()
    
    for bar, value in zip(bars, top_cats['revenue']):
        plt.text(bar.get_width() + top_cats['revenue'].max() * 0.01,
                 bar.get_y() + bar.get_height() / 2,
                 f'${value:,.0f}', va='center', fontsize=9)
    
    plt.tight_layout()
    plt.show()

## Part 7: Advanced Insights - Customer Segmentation

In [ ]:
# RFM Segmentation
print('=' * 60)
print('CUSTOMER SEGMENTATION (RFM Analysis)')
print('=' * 60)

rfm = perform_rfm_segmentation(df)

if not rfm.empty:
    print('\nRFM Segment Distribution:')
    segment_summary = rfm['segment'].value_counts()
    for segment, count in segment_summary.items():
        pct = count / len(rfm) * 100
        print(f"  {segment}: {count:,} ({pct:.1f}%)")

In [ ]:
# Visualize RFM segments
if not rfm.empty:
    plt.figure(figsize=(12, 6))
    segment_counts = rfm['segment'].value_counts()
    
    colors = sns.color_palette('Set3', len(segment_counts))
    bars = plt.bar(segment_counts.index, segment_counts.values, color=colors)
    
    plt.xlabel('Customer Segment', fontsize=12)
    plt.ylabel('Number of Customers', fontsize=12)
    plt.title('Customer Distribution by RFM Segment', fontsize=14, fontweight='bold')
    plt.xticks(rotation=45)
    
    for bar, count in zip(bars, segment_counts.values):
        plt.text(bar.get_x() + bar.get_width() / 2,
                 bar.get_height() + segment_counts.max() * 0.01,
                 f'{count:,}', ha='center', fontsize=9)
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Customer Lifetime Value
print('\nCustomer Lifetime Value Analysis:')
clv = calculate_customer_lifetime_value(df, months_ahead=12)

if not clv.empty:
    print(f"\nAverage Predicted CLV (12 months): ${clv['predicted_clv'].mean():,.2f}")
    print(f"Max Predicted CLV: ${clv['predicted_clv'].max():,.2f}")
    print(f"Median Predicted CLV: ${clv['predicted_clv'].median():,.2f}")

In [ ]:
# Churn Risk Analysis
print('\nChurn Risk Analysis:')
churn = calculate_churn_risk(df, days_threshold=90)

if not churn.empty:
    churn_summary = churn['churn_risk'].value_counts()
    print('\nChurn Risk Distribution:')
    for risk, count in churn_summary.items():
        pct = count / len(churn) * 100
        print(f"  {risk} Risk: {count:,} ({pct:.1f}%)")

## Part 8: Sales Forecasting

In [ ]:
# Sales Forecast
print('=' * 60)
print('SALES FORECAST (Next 3 Months)')
print('=' * 60)

forecast = forecast_sales(df, periods=3)
forecast_data = forecast[forecast['is_forecast'] == True]

if not forecast_data.empty:
    print('\nProjected Monthly Revenue:')
    for _, row in forecast_data.iterrows():
        print(f"  {row['month']}: ${row['sales']:,.2f}")

In [ ]:
# Visualize forecast
if not forecast.empty:
    plt.figure(figsize=(12, 6))
    
    historical = forecast[forecast['is_forecast'] == False]
    predicted = forecast[forecast['is_forecast'] == True]
    
    plt.plot(historical['month'].astype(str), historical['sales'], 
             marker='o', linewidth=2, label='Historical', color='#2E86AB')
    plt.plot(predicted['month'].astype(str), predicted['sales'], 
             marker='s', linewidth=2, label='Forecasted', color='#DC3545', linestyle='--')
    
    plt.xlabel('Month', fontsize=12)
    plt.ylabel('Revenue ($)', fontsize=12)
    plt.title('Sales Forecast', fontsize=14, fontweight='bold')
    plt.xticks(rotation=45)
    plt.legend()
    plt.gca().yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x:,.0f}'))
    plt.tight_layout()
    plt.show()

## Part 9: Product Affinity Analysis

In [ ]:
# Product Affinity
print('=' * 60)
print('PRODUCT AFFINITY ANALYSIS')
print('=' * 60)

affinity = analyze_product_affinity(df)

if not affinity.empty:
    print('\nTop 10 Product Pairs Bought Together:')
    for _, row in affinity.head(10).iterrows():
        print(f"  Products {row['product_1']} + {row['product_2']}: {row['co_occurrence']} orders")
else:
    print('\nInsufficient data for affinity analysis')
    print('(Need orders with multiple different products)')

## Part 10: Customer & Delivery Analysis

In [ ]:
# Customer Analysis
print('=' * 60)
print('CUSTOMER ANALYSIS')
print('=' * 60)

customer_results = analyze_customers(df)

print(f"\nUnique Customers: {customer_results['unique_customers']:,}")
print(f"Repeat Customers: {customer_results['repeat_customers']:,}")
print(f"New Customers: {customer_results['new_customers']:,}")
print(f"Customer Repeat Rate: {customer_results['repeat_rate']*100:.1f}%")

In [ ]:
# Delivery Analysis
print('\n' + '=' * 60)
print('DELIVERY ANALYSIS')
print('=' * 60)

delivery_results = analyze_delivery(df)

print(f"\nAverage Delivery Time: {delivery_results['avg_delivery_time']:.1f} days")
print(f"Median Delivery Time: {delivery_results['median_delivery_time']:.1f} days")
print(f"Delay Rate: {delivery_results['delay_rate']*100:.1f}%")
print(f"\nDelivery-Review Correlation: {delivery_results['delivery_review_correlation']:.3f}")

In [ ]:
# Delivery Time Distribution
if len(delivery_results.get('delivery_times', [])) > 0:
    plt.figure(figsize=(12, 6))
    
    delivery_times = delivery_results['delivery_times']
    
    sns.histplot(delivery_times, kde=True, color='#28A745', bins=30)
    
    plt.axvline(delivery_results['avg_delivery_time'], color='red', 
                linestyle='--', linewidth=2, 
                label=f"Mean: {delivery_results['avg_delivery_time']:.1f} days")
    plt.axvline(delivery_results['median_delivery_time'], color='blue', 
                linestyle='--', linewidth=2,
                label=f"Median: {delivery_results['median_delivery_time']:.1f} days")
    
    plt.xlabel('Delivery Time (days)', fontsize=12)
    plt.ylabel('Number of Orders', fontsize=12)
    plt.title('Delivery Time Distribution', fontsize=14, fontweight='bold')
    plt.legend()
    plt.tight_layout()
    plt.show()

## Part 11: Payment Analysis

In [ ]:
# Payment Analysis
print('=' * 60)
print('PAYMENT ANALYSIS')
print('=' * 60)

payment_results = analyze_payments(df)

print('\nPayment Method Distribution:')
if not payment_results['payment_type_distribution'].empty:
    for idx, row in payment_results['payment_type_distribution'].iterrows():
        print(f"  - {row['payment_type']}: {row['percentage']:.1f}%")

print(f"\nAverage Installments: {payment_results['avg_installments']:.1f}")
print(f"Orders Using Installments: {payment_results['installment_rate']*100:.1f}%")

In [ ]:
# Payment Type Distribution
if not payment_results['payment_type_distribution'].empty:
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    payment_counts = df['payment_type'].value_counts()
    colors = sns.color_palette('Set2', len(payment_counts))
    
    wedges, texts, autotexts = axes[0].pie(
        payment_counts.values,
        labels=[pt.replace('_', ' ').title() for pt in payment_counts.index],
        autopct='%1.1f%%',
        colors=colors,
        startangle=90
    )
    
    for autotext in autotexts:
        autotext.set_fontsize(10)
        autotext.set_fontweight('bold')
    
    axes[0].set_title('Payment Method Distribution', fontsize=14, fontweight='bold')
    
    if not payment_results['payment_value_by_type'].empty:
        pv = payment_results['payment_value_by_type']
        bars = axes[1].bar(
            [pt.replace('_', ' ').title() for pt in pv['payment_type']],
            pv['total_value'],
            color=sns.color_palette('coolwarm', len(pv))
        )
        
        axes[1].set_xlabel('Payment Type', fontsize=12)
        axes[1].set_ylabel('Total Value ($)', fontsize=12)
        axes[1].set_title('Payment Value by Method', fontsize=14, fontweight='bold')
        axes[1].tick_params(axis='x', rotation=45)
        axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x:,.0f}'))
    
    plt.tight_layout()
    plt.show()

## Part 12: Generate Static Visualizations

In [ ]:
# Save all static charts
print('=' * 60)
print('SAVING VISUALIZATIONS')
print('=' * 60)

charts_dir = os.path.join(OUTPUT_DIR, 'charts')

all_results = {
    'sales': sales_results,
    'customers': customer_results,
    'products': analyze_products(df),
    'delivery': delivery_results,
    'payments': payment_results
}

saved_charts = save_all_charts(df, charts_dir, all_results)

print('\nSaved charts:')
for name, path in saved_charts.items():
    print(f"  - {name}: {path}")

## Part 13: Generate Interactive Dashboards

In [ ]:
# Create interactive dashboards
print('=' * 60)
print('GENERATING INTERACTIVE DASHBOARDS')
print('=' * 60)

dashboard_dir = os.path.join(OUTPUT_DIR, 'dashboards')

saved_dashboards = create_full_dashboard(df, kpis, dashboard_dir)

print('\nInteractive dashboards saved:')
for name, path in saved_dashboards.items():
    print(f"  - {name}: {path}")

print('\nOpen any HTML file in a browser to view interactive dashboards!')

## Part 14: Generate Reports

In [ ]:
# Generate executive summary
print('=' * 60)
print('GENERATING REPORTS')
print('=' * 60)

reports_dir = os.path.join(OUTPUT_DIR, 'reports')
os.makedirs(reports_dir, exist_ok=True)

# Executive summary
executive_report_path = os.path.join(reports_dir, 'executive_summary.txt')
save_executive_report(df, executive_report_path)

# Standard insights report
standard_report_path = os.path.join(reports_dir, 'summary.txt')
standard_report = generate_insights(all_results)
with open(standard_report_path, 'w') as f:
    f.write(standard_report)
print(f'Standard report saved to: {standard_report_path}')

# Advanced insights report
advanced_report_path = os.path.join(reports_dir, 'advanced_insights.txt')
advanced_report = generate_advanced_insights_report(df)
with open(advanced_report_path, 'w') as f:
    f.write(advanced_report)
print(f'Advanced insights saved to: {advanced_report_path}')

In [ ]:
# Display executive summary
executive_summary = generate_executive_summary(df, kpis)
print(executive_summary)

## Part 15: Final Summary

In [ ]:
# Final summary
print('=' * 70)
print('ANALYSIS COMPLETE - SUMMARY')
print('=' * 70)

summary = f"""
Dataset Statistics:
  - Total Records: {len(df):,}
  - Unique Orders: {df['order_id'].nunique():,}
  - Unique Customers: {df['customer_id'].nunique():,}
  - Unique Products: {df['product_id'].nunique():,}
  - Date Range: {df['order_purchase_timestamp'].min().strftime('%Y-%m-%d')} to {df['order_purchase_timestamp'].max().strftime('%Y-%m-%d')}

Key Metrics:
  - Total Revenue: ${kpis['revenue']['total_revenue']:,.2f}
  - Average Order Value: ${kpis['revenue']['avg_ticket_value']:.2f}
  - Customer Repeat Rate: {kpis['customers']['repeat_customer_rate']:.1f}%
  - Average Delivery Time: {kpis['delivery']['avg_delivery_time_days']:.1f} days
  - On-Time Delivery Rate: {kpis['delivery']['on_time_delivery_rate']:.1f}%
  - Average Review Score: {kpis['orders']['avg_review_score']:.2f}/5.0

Outputs Generated:
  - Static Charts: {len(saved_charts)} files in {charts_dir}
  - Interactive Dashboards: {len(saved_dashboards)} files in {dashboard_dir}
  - Reports:
    * Executive Summary: {executive_report_path}
    * Standard Report: {standard_report_path}
    * Advanced Insights: {advanced_report_path}

Advanced Analytics:
  - RFM Customer Segmentation: {len(rfm)} customers segmented
  - Churn Risk Assessment: {len(churn)} customers analyzed
  - Sales Forecast: 3-month projection generated
  - Product Affinity: {len(affinity)} product pairs analyzed
"""

print(summary)

---
## Analysis Complete!

### Generated Outputs:

**Static Charts** (`outputs/charts/`):
- Revenue trend line chart
- Top categories bar chart
- Payment types pie/bar chart
- Correlation heatmap
- Delivery time histogram
- Customer geography chart

**Interactive Dashboards** (`outputs/dashboards/`):
- Revenue Dashboard (HTML)
- Customer Dashboard (HTML)
- Product Dashboard (HTML)
- Delivery Dashboard (HTML)
- Payment Dashboard (HTML)
- Executive Summary (HTML)

**Reports** (`outputs/reports/`):
- Executive Summary (TXT)
- Standard Analysis Report (TXT)
- Advanced Insights Report (TXT)

### Next Steps:
1. Open interactive dashboards in a browser for exploration
2. Review executive summary for key insights and recommendations
3. Share findings with stakeholders
4. Implement recommended actions based on KPI alerts